# Notebook 02: Evaluation Analysis
Analyze RAG evaluation results, identify failure modes, and improve the pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.evaluation.metrics import (
    EvaluationSample,
    StatisticalMetrics,
)
from src.evaluation.evaluator import RAGEvaluator, EvaluationReport

print('Imports OK')

In [ ]:
# Load evaluation report if it exists
report_path = Path('../data/eval/report.json')

if report_path.exists():
    with open(report_path) as f:
        report_data = json.load(f)
    
    summary = report_data['summary']
    results = report_data['results']
    
    print(f"Dataset: {summary['dataset_name']}")
    print(f"Total samples: {summary['total_samples']}")
    print(f"Overall pass rate: {summary['overall_pass_rate']:.1%}")
    print()
    print('Metric Averages:')
    for metric, avg in summary['metric_averages'].items():
        print(f'  {metric}: {avg:.3f}')
else:
    print('No report found. Run: make evaluate')

In [ ]:
# Run quick statistical evaluation on sample data
stat_metrics = StatisticalMetrics()

test_samples = [
    EvaluationSample(
        question='How do I reset my password?',
        answer='Go to login page, click Forgot Password, enter your email.',
        contexts=['Click Forgot Password to reset your password via email.'],
        ground_truth='Click Forgot Password on the login page.',
    ),
    EvaluationSample(
        question='What is the refund policy?',
        answer='We offer 30-day returns on all items.',
        contexts=['30-day money back guarantee on physical products.'],
        ground_truth='30-day money back guarantee on physical products.',
    ),
]

evaluator = RAGEvaluator(
    use_llm_metrics=False,       # Skip LLM metrics for quick analysis
    use_statistical_metrics=True,
)

report = evaluator.evaluate_answers(test_samples, dataset_name='notebook_eval')
report.print_summary()

In [ ]:
# Visualize metric scores
if report.metric_averages:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    metrics = list(report.metric_averages.keys())
    averages = list(report.metric_averages.values())
    pass_rates = [report.pass_rates.get(m, 0) for m in metrics]
    
    # Bar chart of averages
    colors = ['#2ecc71' if v >= 0.7 else '#e74c3c' for v in averages]
    ax1.barh(metrics, averages, color=colors)
    ax1.axvline(x=0.7, color='orange', linestyle='--', label='Threshold (0.7)')
    ax1.set_xlim(0, 1)
    ax1.set_title('Average Metric Scores')
    ax1.set_xlabel('Score')
    ax1.legend()
    
    # Bar chart of pass rates
    colors2 = ['#2ecc71' if v >= 0.8 else '#e74c3c' for v in pass_rates]
    ax2.barh(metrics, pass_rates, color=colors2)
    ax2.axvline(x=0.8, color='orange', linestyle='--', label='Target (80%)')
    ax2.set_xlim(0, 1)
    ax2.set_title('Pass Rates by Metric')
    ax2.set_xlabel('Pass Rate')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('../data/eval/metric_analysis.png', dpi=150)
    plt.show()